1. Veriyi Nereye Koymalısın?
İndirdiğin .xlsx dosyasını projenin en kök dizinindeki data/raw/ klasörüne yerleştir.

data/raw/: Buradaki verilere asla dokunma. Bu senin "altın kaynağın". Bir hata yaparsan hep buraya geri dönebilirsin.

data/processed/: Analiz ettikten, temizledikten ve formatını değiştirdikten sonraki halini buraya kaydedeceğiz.

2. XLSX Dosyasını Python ile Açma ve Dönüştürme
Python'da Excel dosyalarını okumak için en standart yol pandas kütüphanesidir. Arka planda Excel'i anlaması için openpyxl kütüphanesine de ihtiyacın olacak.

Python
import pandas as pd

 Excel dosyasını oku
df = pd.read_excel('data/raw/verin.xlsx', engine='openpyxl')

 İlk bakış
print(df.head())
print(df.info()) # Kolon tipleri ve boş değerleri görmek için kritik
3. CSV mi, Parquet mi?
Analiz ve MLOps süreçlerinde format seçimi önemlidir:

CSV: İnsan tarafından okunabilir, her yerde açılır. Ancak büyük verilerde yavaştır ve veri tiplerini (tarih, sayı vb.) kaybedebilir; her seferinde tekrar tanımlaman gerekir.

Parquet: ML dünyasının standardıdır. Sıkıştırılmış bir format olduğu için diskte çok az yer kaplar ve çok hızlı okunur. En önemlisi, veri tiplerini (şema) içinde saklar.

Tavsiye: Ham verin .xlsx kalsın, ancak analiz ve modelleme aşamasında veriyi temizledikten sonra data/processed/ klasörüne .parquet olarak kaydet.

Python
 Temizlenmiş veriyi Parquet olarak kaydetme
df.to_parquet('data/processed/cleaned_data.parquet')
4. Analiz (EDA) Nasıl Yapılır?
ERP ve talep tahmini projelerinde analiz yaparken şu adımları izlemelisin:

Zaman Serisi Kontrolü: Tarih kolonu datetime tipinde mi? Veride eksik gün var mı?

Eksik Veri (Missing Values): Üretim verisinde boşluklar genelde "o gün sipariş yok" anlamına gelir. Bunu 0 ile mi doldurmalısın yoksa ortalama ile mi?

Aykırı Değer (Outlier): Olağanüstü büyük bir sipariş gelmiş mi? Bu modelini yanıltabilir.

Korelasyon: Hangi ürünler beraber satılıyor? Mevsimsellik (aylar, bayramlar) satışları nasıl etkiliyor?

In [3]:
import pandas as pd

# Excel dosyasını oku
df = pd.read_excel('../data/raw/erp.xlsx', engine='openpyxl')

# İlk bakış
print(df.head())
print(df.info()) # Kolon tipleri ve boş değerleri görmek için kritik

       Order ID Order Date Origin Port Carrier  TPT Service Level  \
0  1.447296e+09 2013-05-26      PORT09   V44_3    1           CRF   
1  1.447158e+09 2013-05-26      PORT09   V44_3    1           CRF   
2  1.447139e+09 2013-05-26      PORT09   V44_3    1           CRF   
3  1.447364e+09 2013-05-26      PORT09   V44_3    1           CRF   
4  1.447364e+09 2013-05-26      PORT09   V44_3    1           CRF   

   Ship ahead day count  Ship Late Day count   Customer  Product ID  \
0                     3                    0  V55555_53     1700106   
1                     3                    0  V55555_53     1700106   
2                     3                    0  V55555_53     1700106   
3                     3                    0  V55555_53     1700106   
4                     3                    0  V55555_53     1700106   

  Plant Code Destination Port  Unit quantity  Weight  
0    PLANT16           PORT09            808   14.30  
1    PLANT16           PORT09           3188   8

In [4]:
# Verinin başlangıç ve bitiş tarihlerini görelim
print("İlk Sipariş Tarihi:", df['Order Date'].min())
print("Son Sipariş Tarihi:", df['Order Date'].max())

# Kaç farklı ürün ve müşteri var?
print("\nBenzersiz Ürün (Product ID) Sayısı:", df['Product ID'].nunique())
print("Benzersiz Müşteri Sayısı:", df['Customer'].nunique())

# Ürünlerin genel dağılımına (Miktar bazında) hızlı bir bakış
print("\nToplam Sipariş Miktarı (Unit quantity) İstatistikleri:")
print(df['Unit quantity'].describe())

İlk Sipariş Tarihi: 2013-05-26 00:00:00
Son Sipariş Tarihi: 2013-05-26 00:00:00

Benzersiz Ürün (Product ID) Sayısı: 772
Benzersiz Müşteri Sayısı: 46

Toplam Sipariş Miktarı (Unit quantity) İstatistikleri:
count      9215.000000
mean       3202.747151
std       15965.622260
min         235.000000
25%         330.000000
50%         477.000000
75%        1275.500000
max      561847.000000
Name: Unit quantity, dtype: float64


In [4]:
import os
import zipfile
import pandas as pd

# Hedef klasörleri belirle
RAW_DATA_PATH = '../data/raw/'

# 3. CSV dosyalarını Pandas ile oku
train = pd.read_csv(os.path.join(RAW_DATA_PATH, 'train.csv'), parse_dates=['date'])
test = pd.read_csv(os.path.join(RAW_DATA_PATH, 'test.csv'), parse_dates=['date'])

# Önişleme için birleştir
df = pd.concat([train, test], sort=False)
print(df.head())

        date  store  item  sales  id
0 2013-01-01      1     1   13.0 NaN
1 2013-01-02      1     1   11.0 NaN
2 2013-01-03      1     1   14.0 NaN
3 2013-01-04      1     1   13.0 NaN
4 2013-01-05      1     1   10.0 NaN


In [5]:
# Verinin başlangıç ve bitiş tarihlerini görelim
print("İlk Sipariş Tarihi:", df['date'].min())
print("Son Sipariş Tarihi:", df['date'].max())


İlk Sipariş Tarihi: 2013-01-01 00:00:00
Son Sipariş Tarihi: 2018-03-31 00:00:00


In [6]:
def create_date_features(df):
    df['month'] = df.date.dt.month
    df['day_of_month'] = df.date.dt.day
    df['day_of_year'] = df.date.dt.dayofyear
    df['week_of_year'] = df.date.dt.isocalendar().week
    df['day_of_week'] = df.date.dt.dayofweek
    df['year'] = df.date.dt.year
    df['is_wknd'] = df.date.dt.dayofweek // 5  # Cumartesi(5) ve Pazar(6) için 1 üretir, diğerleri 0
    df['is_month_start'] = df.date.dt.is_month_start.astype(int)
    df['is_month_end'] = df.date.dt.is_month_end.astype(int)
    return df

df = create_date_features(df)
print(df.head())

        date  store  item  sales  id  month  day_of_month  day_of_year  \
0 2013-01-01      1     1   13.0 NaN      1             1            1   
1 2013-01-02      1     1   11.0 NaN      1             2            2   
2 2013-01-03      1     1   14.0 NaN      1             3            3   
3 2013-01-04      1     1   13.0 NaN      1             4            4   
4 2013-01-05      1     1   10.0 NaN      1             5            5   

   week_of_year  day_of_week  year  is_wknd  is_month_start  is_month_end  
0             1            1  2013        0               1             0  
1             1            2  2013        0               0             0  
2             1            3  2013        0               0             0  
3             1            4  2013        0               0             0  
4             1            5  2013        1               0             0  


In [ ]:
# ÇOK ÖNEMLİ: Zaman serisinde geçmişi hesaplamadan önce veriyi tarihe göre sıralamalısın!
df.sort_values(by=['store', 'item', 'date'], axis=0, inplace=True)

def lag_features(dataframe, lags):
    for lag in lags:
        # Satış değerlerini 'lag' gün sayısı kadar aşağı kaydırıp yeni kolon yapıyoruz
        dataframe['sales_lag_' + str(lag)] = dataframe.groupby(["store", "item"])['sales'].transform(
            lambda x: x.shift(lag))
    return dataframe
 
# ERP ve perakende mantığında anlamlı gecikmeler:
# 91 gün (1 çeyrek), 98 (14 hafta), 182 (Yarım yıl), 364 (1 Yıl)
df = lag_features(df, [91, 98, 105, 112, 119, 126, 182, 364])

Model geleceği tahmin edecekse, geçmişte ne olduğunu bilmek zorundadır. "Bu ürün, bu mağazada 1 hafta önce, 3 ay önce, 1 yıl önce ne kadar sattı?" sorusunun cevabını ayrı kolonlar olarak ekleriz.
sales_lag_91: "Bugünden tam 91 gün geriye git. O günkü takvim yaprağında tam olarak kaç adet satılmış? Sadece o günün rakamını getir."

groupby(["store", "item"])['sales']: Bu kısım hayati önem taşır. Elimizde 10 mağaza ve 50 ürün var. Bütün veriler alt alta duruyor. Eğer "satışları 91 gün aşağı kaydır" dersek ve gruplama yapmazsak, Mağaza 1'deki Ürün 50'nin satışı, Mağaza 2'deki Ürün 1'in hizasına kayabilir. Bu yüzden "Her mağazayı ve ürünü kendi içinde paketle, sadece kendi geçmişini aşağı kaydır" diyoruz.

.shift(lag): Excel'deki hücreleri tutup aşağı çekmek gibidir. lag=91 ise, bugünkü satırın yanına, 91 satır (91 gün) önceki satışı yazar. İlk 91 günün karşısı doğal olarak NaN (boş) kalır.

.transform(lambda x: ...): groupby yaptığında tablo parçalanır. transform, bu parçalanmış tablodaki hesaplamayı yapar ve orijinal ana tablonun satır sayısını bozmadan sonucu yeni bir kolon olarak geri yapıştırır.


In [ ]:
def roll_mean_features(dataframe, windows):
    for window in windows:
        dataframe['sales_roll_mean_' + str(window)] = dataframe.groupby(["store", "item"])['sales']. \
                                                          transform(
            lambda x: x.shift(1).rolling(window=window, min_periods=10, win_type="triang").mean())
    return dataframe

# 3 aylık, 6 aylık, 1 yıllık ortalamalar
df = roll_mean_features(df, [365, 546])

x.shift(1) (Mülakat Sırrı): "Neden ortalama almadan önce veriyi 1 gün kaydırdın?" diye sorarlarsa cevabın şu olmalı: "Çünkü Data Leakage (Veri Sızıntısı) olmasını engellemek istedim. Model yarının satışını tahmin ederken yarının gerçek satış bilgisini bilemez. Ortalamanın içine tahmin etmeye çalıştığımız günün satışını dahil etmemek için veriyi önce 1 gün geriye kaydırdım, sonra geçmişin ortalamasını aldım."

.rolling(window=365): Geriye dönük 365 günlük bir pencere açar ve her gün için bu pencereyi kaydırarak ilerler.

min_periods=10: İlk günlerde 365 günlük geçmiş yoktur. "Geçmişte en az 10 günlük veri varsa bile ortalama hesapla, yoksa NaN yaz" demek için kullanılır.

win_type="triang" (Üçgen Pencere): Dümdüz ortalama almak yerine, "Yakın geçmiş (örneğin dünün satışı), uzak geçmişten (örneğin 300 gün önceki satış) daha önemlidir" diyerek yakın günlere daha çok matematiksel ağırlık verir.

In [15]:
def ewm_features(dataframe, alphas, lags):
    for alpha in alphas:
        for lag in lags:
            # alpha değeri ne kadar büyükse yakın geçmiş o kadar önemlidir
            dataframe['sales_ewm_alpha_' + str(alpha).replace(".", "") + "_lag_" + str(lag)] = \
                dataframe.groupby(["store", "item"])['sales'].transform(lambda x: x.shift(lag).ewm(alpha=alpha).mean())
    return dataframe

# alpha: 0.95 (düne çok güven), 0.1 (geçmişe daha çok güven)
alphas = [0.95, 0.9, 0.8, 0.7, 0.5]
lags = [91, 98, 105, 112, 180, 270, 365, 546]

df = ewm_features(df, alphas, lags)

Normal "Rolling Mean" (Hareketli Ortalama) adil bir demokrasidir.

Mesela 10 günlük ortalama aldığında; dünün satışının da, 10 gün önceki satışın da oylamadaki gücü eşittir (ikisi de %10 etkiler).

**Ama gerçek hayatta (ve ERP sistemlerinde) dün olanlar, 10 gün önce olanlardan daha önemlidir!** EWM (Exponentially Weighted Mean) işte bu gerçeği makineye öğretir.

EWM'de geçmişe doğru gittikçe günlerin "önemi" üssel olarak (hızla) düşer. Bunu belirleyen şey **Alpha ($\alpha$)** değeridir. Alpha 0 ile 1 arasında bir değer alır:

- **Alpha = 0.95 (Balık Hafızalı Model):** *"Sadece düne ve ondan önceki güne çok güveniyorum, 1 hafta öncesi umurumda değil"* der. Eğer dün satışlar aniden patladıysa (örneğin bir ürün viral olduysa), EWM bunu anında fark edip yarının tahminini hemen yükseltir. Normal ortalama ise bu patlamayı fark edene kadar günler geçer.
- **Alpha = 0.1 (Fil Hafızalı Model):** *"Dünkü ani satışlara hemen kanmam, ben uzun vadeli geçmişe bakarım"* der.

In [16]:
# Verinin tarihe göre sıralı olduğundan emin olalım (Her zaman ilk kural!)
df.sort_values(by=['store', 'item', 'date'], axis=0, inplace=True)

# 1. Hız ve İvme (Trend) Özellikleri
# 91 gün (1 çeyrek) önceki satış ile, ondan 1 hafta öncesi (98) arasındaki fark nedir?
# Eğer sonuç pozitifse, satışlar o dönemde artış trendine girmiş demektir.
df['sales_trend_1week'] = df['sales_lag_91'] - df['sales_lag_98']

# 91 gün öncesi ile 1 ay öncesi (119) arasındaki fark
df['sales_trend_1month'] = df['sales_lag_91'] - df['sales_lag_119']


# 2. Fiyat / Ürün Kalitesi Simülasyonu (Mean Encoding)
# "Bu ürün tüm mağazalarda genel olarak popüler bir ürün mü?"
# Her ürünün tüm zamanlardaki ortalama satışını bulup bir kolon olarak ekliyoruz.
# (Transform'u burada tam olarak sınıf ortalaması örneğindeki gibi kullanıyoruz!)
df['item_general_popularity'] = df.groupby('item')['sales'].transform('mean')

# "Bu mağaza genel olarak kalabalık/çok satan bir mağaza mı?"
df['store_general_crowd'] = df.groupby('store')['sales'].transform('mean')

# Günlerin popülaritesi (Pazartesileri genel olarak iş yapar mı?)
df['day_of_week_popularity'] = df.groupby('day_of_week')['sales'].transform('mean')

Veri setinde sadece tarih ve miktar kolonları vardı. Ben ham veriyi doğrudan algoritmaya vermek yerine, ERP ve MRP (Malzeme İhtiyaç Planlaması) mantığına uygun bir özellik mühendisliği (feature engineering) yaptım. İş zekası (Business Intelligence) kurallarını kullanarak modele geçmiş alışveriş davranışlarını (Lag) ve trendleri (Rolling) kolon olarak ekledim. Modelin ezberlemesini değil, mevsimsel periyodu öğrenmesini sağladım

In [17]:
df.head(10)

,date,store,item,sales,id,month,day_of_month,day_of_year,week_of_year,day_of_week,...,sales_ewm_alpha_05_lag_112,sales_ewm_alpha_05_lag_180,sales_ewm_alpha_05_lag_270,sales_ewm_alpha_05_lag_365,sales_ewm_alpha_05_lag_546,sales_trend_1week,sales_trend_1month,item_general_popularity,store_general_crowd,day_of_week_popularity
0,2013-01-01,1,1,13.0,NaN,1,1,1,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.981599,47.268379,48.225908
1,2013-01-02,1,1,11.0,NaN,1,2,2,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.981599,47.268379,48.368506
2,2013-01-03,1,1,14.0,NaN,1,3,3,1,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.981599,47.268379,51.723218
3,2013-01-04,1,1,13.0,NaN,1,4,4,1,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.981599,47.268379,55.157249
4,2013-01-05,1,1,10.0,NaN,1,5,5,1,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.981599,47.268379,58.662697
5,2013-01-06,1,1,12.0,NaN,1,6,6,1,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.981599,47.268379,62.143333
6,2013-01-07,1,1,10.0,NaN,1,7,7,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.981599,47.268379,41.429638
7,2013-01-08,1,1,9.0,NaN,1,8,8,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.981599,47.268379,48.225908
8,2013-01-09,1,1,12.0,NaN,1,9,9,2,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.981599,47.268379,48.368506
9,2013-01-10,1,1,9.0,NaN,1,10,10,2,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.981599,47.268379,51.723218


In [27]:
df.tail(10)

,date,store,item,sales,id,month,day_of_month,day_of_year,week_of_year,day_of_week,...,sales_ewm_alpha_05_lag_112,sales_ewm_alpha_05_lag_180,sales_ewm_alpha_05_lag_270,sales_ewm_alpha_05_lag_365,sales_ewm_alpha_05_lag_546,sales_trend_1week,sales_trend_1month,item_general_popularity,store_general_crowd,day_of_week_popularity
44990,2018-03-22,10,50,NaN,44990.0,3,22,81,12,3,...,74.722289,93.517970,109.161929,62.518663,83.648400,-9.0,-24.0,65.882202,58.709288,51.723218
44991,2018-03-23,10,50,NaN,44991.0,3,23,82,12,4,...,70.361145,96.258985,95.580964,64.759332,82.824200,3.0,-6.0,65.882202,58.709288,55.157249
44992,2018-03-24,10,50,NaN,44992.0,3,24,83,12,5,...,59.680572,79.629492,89.290482,68.879666,89.912100,18.0,-8.0,65.882202,58.709288,58.662697
44993,2018-03-25,10,50,NaN,44993.0,3,25,84,12,6,...,67.340286,79.314746,90.145241,73.939833,101.956050,-10.0,-37.0,65.882202,58.709288,62.143333
44994,2018-03-26,10,50,NaN,44994.0,3,26,85,13,0,...,60.670143,79.657373,106.072621,77.469916,73.978025,-2.0,-24.0,65.882202,58.709288,41.429638
44995,2018-03-27,10,50,NaN,44995.0,3,27,86,13,1,...,64.335072,80.828687,109.036310,71.734958,85.489012,-13.0,-27.0,65.882202,58.709288,48.225908
44996,2018-03-28,10,50,NaN,44996.0,3,28,87,13,2,...,62.167536,85.414343,114.018155,65.867479,82.744506,12.0,-13.0,65.882202,58.709288,48.368506
44997,2018-03-29,10,50,NaN,44997.0,3,29,88,13,3,...,64.083768,94.207172,117.009078,69.433740,90.872253,-4.0,-14.0,65.882202,58.709288,51.723218
44998,2018-03-30,10,50,NaN,44998.0,3,30,89,13,4,...,65.541884,96.603586,108.004539,68.716870,84.936127,-1.0,8.0,65.882202,58.709288,55.157249
44999,2018-03-31,10,50,NaN,44999.0,3,31,90,13,5,...,67.270942,83.801793,103.002269,68.858435,90.968063,-8.0,13.0,65.882202,58.709288,58.662697


In [26]:
a=df.isnull().sum()
a[a>0]

sales                           45000
id                             913000
sales_lag_91                    45500
sales_lag_98                    49000
sales_lag_105                   52500
sales_lag_112                   56000
sales_lag_119                   59500
sales_lag_126                   63000
sales_lag_182                   91000
sales_lag_364                  182000
sales_ewm_alpha_095_lag_91      45500
sales_ewm_alpha_095_lag_98      49000
sales_ewm_alpha_095_lag_105     52500
sales_ewm_alpha_095_lag_112     56000
sales_ewm_alpha_095_lag_180     90000
sales_ewm_alpha_095_lag_270    135000
sales_ewm_alpha_095_lag_365    182500
sales_ewm_alpha_095_lag_546    273000
sales_ewm_alpha_09_lag_91       45500
sales_ewm_alpha_09_lag_98       49000
sales_ewm_alpha_09_lag_105      52500
sales_ewm_alpha_09_lag_112      56000
sales_ewm_alpha_09_lag_180      90000
sales_ewm_alpha_09_lag_270     135000
sales_ewm_alpha_09_lag_365     182500
sales_ewm_alpha_09_lag_546     273000
sales_ewm_al

5 Yıllık Veriden 1.5 Yılı (%30'u) Silmek Çok Değil mi?
Klasik bir yazılımcı "Evet çok, verimi silmem" der. Bir Veri Bilimcisi (MLOps) ise şunu bilir: Kalite > Miktar.

Elinde "Geçmişte ne olduğunu hiç bilmeyen, aptal" 5 yıllık bir veri olacağına;

Elinde "Dünü, geçen haftayı, 3 ay önceyi, geçen seneyi sular seller gibi ezbere bilen, 30 kolonluk devasa bir hafızaya sahip" 3.5 yıllık veri olması modeli uçurur. Zaten Kaggle'da birinci olanlar da tam olarak ilk 1-1.5 yılı feda edip bu zengin kolonları üretenlerdir.

1. Kısım: Başlangıç Koordinatı (Milat) ve Mutfaktaki Pandas
Kaggle bize dedi ki: "Al sana veri, ilk gün 1 Ocak 2013." İşte bu tarih bizim Miladımızdır (Sıfır Noktası). Evren 1 Ocak 2013'te var oldu, ondan öncesi karanlık, hiçlik, yok!

Şimdi mutfağa (Jupyter Notebook'a) girdik ve elimizde sadece 5 günlük bir veri var diyelim:

HAM VERİ (Büyük Patlama):

Satır 1: 1 Ocak -> Satış: 10 (Evrenin ilk günü)

Satır 2: 2 Ocak -> Satış: 15

Satır 3: 3 Ocak -> Satış: 20

Satır 4: 4 Ocak -> Satış: 25

Satır 5: 5 Ocak -> Satış: 30

Biz mutfaktayken (daha modeli hiç kurmamışken) Pandas'a şu kodu yazdık:
dataframe['lag_2'] = shift(2) (Yani git 2 gün önceki satışı yan kolona yaz).

MUTFAKTA NELER OLUYOR? (Pandas'ın İşi):

1 Ocak'a bakar: "Bana 2 gün öncesi lazım. 30 Aralık? Hata! Evren 1 Ocak'ta yaratıldı. O zaman buraya NaN yazıyorum."

2 Ocak'a bakar: "Bana 2 gün öncesi lazım. 31 Aralık? Yok! Buraya da NaN yazıyorum."

3 Ocak'a bakar: "2 gün öncesi... Hah! 1 Ocak var. Oradaki satış 10'muş." -> Gider 3 Ocak'ın hizasına 10 yazar.

4 Ocak'a bakar: 2 gün öncesi 2 Ocak. Satış 15'miş. -> 15 yazar.

MUTFAKTAN ÇIKAN YENİ TABLOMUZ:

Satır 1: [1 Ocak | Satış: 10 | Lag_2: NaN] ❌ (Çöp)

Satır 2: [2 Ocak | Satış: 15 | Lag_2: NaN] ❌ (Çöp)

Satır 3: [3 Ocak | Satış: 20 | Lag_2: 10] ✅

Satır 4: [4 Ocak | Satış: 25 | Lag_2: 15] ✅

İşte senin o lag_1000 yazsam hepsi null mu olurdu sorun tam da budur! Elinde 500 günlük veri varken shift(1000) dersen, Pandas 1000 gün geriye gidemeyeceği için BÜTÜN satırlara NaN yazar.

Ve biz son hamlemizi yaparız: dropna() kodunu çalıştırıp o ilk 2 satırı (NaN olanları) acımadan çöpe atarız. Elimizde sadece 3. ve 4. satırlar kalır.

2. Kısım: Restoran (Modelin Gözünden Dünya)
Tabloyu mutfakta hazırladık. Hepsini yatay, dümdüz bir tabağa koyduk ve restoranın salonundaki LightGBM Modeline servis ettik.

Model masada oturuyor. Geçmişi bilmez, tarihi bilmez. Önüne sadece şu tabak (satır) gelir:
👉 [Tarih: 4 Ocak, Ürün: Kalem, Satış: 25, Kolon_X: 15]

Model "Aa dur ben bi 2 gün geriye (2 Ocak'a) gideyim" DEMEZ. Çünkü zaten senin mutfakta hazırladığın Kolon_X (Lag_2) isimli sütunun içinde o 15 yazmaktadır. Model sadece dümdüz okur ve şu matematiksel bağlantıyı kurar:
"Hmm, Kolon_X'te 15 yazıyorsa, asıl satış 25 oluyormuş. Demek ki Kolon_X ile Satış arasında bir artış ilişkisi var."

Modelin bütün dehası bu kadar basittir. O sadece önüne koyduğun yatay sayı dizisine bakar.

3. Kısım: Tüm EDA ve Feature Engineering'in "Mix" Hali (Büyük Resim)
Hadi şimdi o 2 haftadır uğraştığımız bütün o karmaşık datetime, lag, rolling, ewm kodlarının 1 Mayıs 2014 tarihindeki tek bir satırı nasıl bir canavara dönüştürdüğünü adım adım izleyelim.

Adım 1: Ham Veri (Fakir Başlangıç)
Sadece 4 tane cılız kolon var.
[2014-05-01 | Store: 1 | Item: 1 | Satış: 50]

Adım 2: Date Features (Zamanı Parçalama)
Model 2014-05-01'den anlamazdı. Onu parçaladık.
[Ay: 5 | Haftanın Günü: 3 (Perşembe) | Hafta Sonu mu: 0 | Yılın Günü: 121]

Adım 3: Lag Features (Zaman Makinesi)
Mutfakta Pandas ile geçmişi bugüne kopyaladık. (Diyelim ki tam 1 yıl önce 1 Mayıs 2013'te 30 tane satılmıştı).
[Lag_91: 45 | Lag_364: 30] (Model artık 1 yıl önceki satışı dümdüz yatay bir kolondan okuyabiliyor).

Adım 4: Rolling Mean (Ortalama Gidişat)
Sadece o güne değil, geçmişteki genel trende baktık.
[Rolling_Mean_365: 35.5] (Son 1 yılın ortalama satışı).

Adım 5: EWM (Ani Şoklar İçin Üssel Ortalama)
Yakın geçmişe çok önem verdik.
[EWM_Alpha_095: 48.2] (Geçen hafta satışlar fırlamış, model bunu hemen hissetti).

Adım 6: Trend (İvme)
Satışlar artıyor mu azalıyor mu? (Geçen çeyrek ile geçen ayın farkı).
[Trend_1_Month: +5]

🌟 FİNAL: Modelin Restoranda Yediği Tabak (Tek Bir Satır)
Model 1 Mayıs 2014 satırına geldiğinde sadece şu yatay sayı dizisini okur:
[5, 3, 0, 121, 45, 30, 35.5, 48.2, 5, 50]

Bütün o kodları yazmamızın tek bir amacı vardı: Modelin önüne o 50 sayısını (Satış) verirken, yanına geçmişi, trendi ve mevsimselliği fısıldayan o devasa sayı dizisini koyabilmek!

2. Eksik Verilere ve Aykırı Değerlere Neden Bakmadık?
Çok haklı bir ML (Makine Öğrenmesi) refleksi gösteriyorsun. "Hani nerede IQR ile aykırı değerleri silmek? Nerede eksik değerleri ortalama ile doldurmak?" diyorsun. Mülakatta bunu sorarlarsa şu efsanevi 3 cevabı vereceksin:

Eksik Veri (Missing Value) Zaten Yoktu: Kaggle'ın bu veri setinde asıl veride hiç boşluk yoktu. Sadece bizim kendi ürettiğimiz Lag'lar yüzünden 2013 yılı boş kaldı. Onu da zaten ortalamayla dolduramayız (geçmişi uyduramayız), o yüzden direkt sileceğiz.

Aykırı Değerler (Outliers) Ağaç Modellerini Bozmaz: Biz Lineer Regresyon veya Derin Öğrenme (DL) kullanmayacağız. LightGBM adında bir "Karar Ağacı" (Decision Tree) modeli kullanacağız. Karar ağaçları aykırı değerlerden etkilenmez! İsterse o gün 1 milyon satış olsun, model sadece "Bu değer 100'den büyük mü küçük mü?" diye dalgalanmalara bakar. O yüzden aykırı değerleri silmekle vakit kaybetmeyiz.

Logaritmik Dönüşüm (Gizli Silah): Aykırı değerlerin sivriliklerini yumuşatmak için, satış değerlerini dümdüz vermek yerine logaritmasını alacağız. (Bunu aşağıdaki kodda yapacağız).

In [29]:
import numpy as np

# 1. TEST ve TRAIN DİYE İKİYE AYIRALIM
test_df = df[df['sales'].isna()].copy()
train_df = df[df['sales'].notnull()].copy()

# 2. ÖNCE ÇÖP KOLONLARI SİLİYORUZ (Hayat Kurtaran Adım)
# Bütün tabloyu yutan o 'id' kolonundan ve modele yaramayacak 'date' kolonundan kurtuluyoruz.
cols_to_drop = ['id', 'date']
train_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# 3. ŞİMDİ SADECE LAG YÜZÜNDEN OLUŞAN NAN'LARI SİL
# id kolonu gittiği için artık sadece ilk 1.5 yıllık geçmişi olmayan satırlar silinecek.
train_df.dropna(inplace=True)

# 4. HEDEF DEĞİŞKENİNİ (SALES) LOGARİTMAYA ÇEVİR
train_df['sales'] = np.log1p(train_df['sales'].values)

# GÖSTERİ ZAMANI
print("--- GERÇEK TEMİZLİK BİTTİ ---")
print("Eğitim Verisi Satır Sayısı:", train_df.shape[0])
print("Tabloda kalan TOPLAM NaN sayısı:", train_df.isnull().sum().sum())
print("\nİşte Modelin Göreceği Pırıl Pırıl Verinin Başı:")
print(train_df.head(3))

--- GERÇEK TEMİZLİK BİTTİ ---
Eğitim Verisi Satır Sayısı: 640000
Tabloda kalan TOPLAM NaN sayısı: 0

İşte Modelin Göreceği Pırıl Pırıl Verinin Başı:
     store  item     sales  month  day_of_month  day_of_year  week_of_year  \
546      1     1  2.833213      7             1          182            27   
547      1     1  3.218876      7             2          183            27   
548      1     1  3.258097      7             3          184            27   

     day_of_week  year  is_wknd  ...  sales_ewm_alpha_05_lag_112  \
546            1  2014        0  ...                   14.941495   
547            2  2014        0  ...                   19.470748   
548            3  2014        0  ...                   18.235374   

     sales_ewm_alpha_05_lag_180  sales_ewm_alpha_05_lag_270  \
546                   12.380970                   13.820499   
547                   11.690485                   13.910250   
548                   15.845243                   16.955125   

     sales_e

In [30]:
train_df.to_parquet('../data/processed/train_cleaned.parquet')



Tabüler Verinin Kralı: "Derin öğrenme (LSTM vb.) görüntü ve seste çok iyidir ama ERP sistemlerindeki gibi Excel/Tablo tarzı (Tabular) verilerde Karar Ağaçları (Decision Trees) her zaman daha iyi performans verir. Modelin ezberlemesini değil, kuralları öğrenmesini istedim."

Hız ve Donanım Optimizasyonu: "XGBoost çok güçlüdür ama LightGBM veriyi histogramlara bölerek çalışır (Histogram-based). Yüz binlerce satırlık ve 65 kolonluk devasa ERP verisinde XGBoost saatler sürerken, LightGBM dakikalar içinde eğitilir. Şirket sunucularını yormaz."

Leaf-Wise (Yaprak Odaklı) Büyüme (ASIL ŞOV): "Geleneksel algoritmalar ağacı katman katman (Level-wise) büyütür. LightGBM ise hatayı en çok düşürecek spesifik yaprağı bulup oradan asimetrik büyür (Leaf-wise). Bu da tabüler verideki karmaşık ERP ilişkilerini çok daha hızlı kavramasını sağlar."

Notebook'ta kodlar hücre hücre çalışır. Ama Terapi Yazılım gibi bir şirkette veriler her hafta güncellenir. "Her pazar gecesi biri gelip Notebook mu çalıştıracak?" Hayır!

Bizim kuracağımız Pipeline şu anlama gelir: "Veriyi al -> Böl -> Eğit -> Test Et -> MLflow'a Kaydet -> Modeli Çıktı Ver" adımlarının tek bir tuşla (python src/train.py yazılarak) baştan sona otomatik çalışmasıdır.

1. Neden Train / Validation (Val) Diye Böldük? Eskiden Sadece Train/Test Vardı?
Eski projelerinde (Smart City, Görüntü İşleme vb.) veri setini rastgele %80 Train, %20 Test diye bölüyordun çünkü o projelerde ZAMANIN bir önemi yoktu. (Kedi ve köpek resimlerini karıştırıp versen de model kedi köpek ayırmayı öğrenir).

Ama zaman serisinde zamanda yolculuk yapamazsın. Zamanı karıştırırsan model "Gelecekten Kopya" çeker (Data Leakage).

Train (Çalışma Masası): Modelin 2013-2016 yılları arasında ders çalıştığı, kuralları öğrendiği yerdir.

Validation (Deneme Sınavı): Modelin dersi ne kadar anladığını ölçmek için kendi kendimize yaptığımız 2017'nin ilk 3 ayını kapsayan sınavdır. "Modelim geleceği tahmin edebiliyor mu?" sorusunun cevabını burada alırız.

Test (ÖSYM Gerçek Sınav): Kaggle'ın bizden istediği, cevapları bizim bilmediğimiz, 2018 yılına ait o asıl tahmin bölgesidir.

Kritik Mülakat Savunması: "Eğer veriyi rastgele bölseydim, model 2017'nin Haziran ayını eğitim sırasında görüp ezberleyecek, sonra ben 2017'nin Mayıs ayını sorunca harika tahmin edecekti. Ama gerçek hayatta (Terapi Yazılım'da) ERP sistemi benden 'YARIN'ı isteyecek, dünü değil. Bu yüzden modeli sadece geçmişle eğitip, sadece gelecekle doğruladım (Time-Based Split)."

2. O Parametreleri Neye Göre Verdik? Arttırsak/Azaltsak Ne Olurdu?
O gördüğün sözlük (params), modelin "Beyin Ayarlarıdır". Rastgele seçilmedi, günlerce süren deneme-yanılmalar (Hyperparameter Tuning) sonucunda Kaggle şampiyonlarının bulduğu optimum ayarlardır. Tek tek inceleyelim:

learning_rate: 0.05 (Öğrenme Hızı): Modelin her ağaçta attığı "adım büyüklüğüdür".

Artırırsak (örn: 0.5): Model çok hızlı öğrenir, eğitimi 2 saniyede biter ama "Körleme koşan adam" gibi asıl hedefi ıskalar, ezberler (Overfitting).

Azaltırsak (örn: 0.01): Bebek adımlarıyla çok hassas öğrenir. Harika tahmin yapar ama eğitilmesi saatler sürer ve binlerce ağaç (num_boost_round) gerektirir.

feature_fraction: 0.8 (Kolon Gizleme): "Her ağaç kurulurken, 65 kolonun hepsine bakma, sadece rastgele %80'ine bak" der.

Neden Yaptık? Model sadece "Lag_364" veya "Trend" gibi 1-2 güçlü kolona bağımlı kalmasın, diğer kolonların da gücünü keşfetsin diye. Modeli farklı açılardan bakmaya zorlar, ezberi (overfitting) bozar.

max_depth: 6 ve num_leaves: 63: Karar ağacının ne kadar derinleşebileceği (kaç soru sorabileceği) ve yaprak sayısıdır.

Artırırsak: Ağaç çok büyür, verideki her ufak detayı (hatta hatalı aykırı değerleri bile) kural sanıp ezberler.

Azaltırsak: Ağaç çok sığ kalır (underfitting), verideki karmaşık ERP/Tedarik mantığını öğrenemez.

Bu "Öğrenme Hızı" ve "Ağaç Sayısı" arasındaki ölümcül dengeyi gözlerinle görmek için şu simülatörü hazırladım:


Görselleştirmeyi göster


3. Neden num_boost_round=1500?
Modelimize "Arka arkaya maksimum 1500 tane ağaç kur" dedik.
"Oha çok değil mi?" diyebilirsin. Ama kodun hemen altında şu sihirli satır vardı:
early_stopping(stopping_rounds=50)

Mantığı: "1500 ağaca kadar yolun var. Ama eğer ağaçları kurarken, son 50 ağaç boyunca Doğrulama Sınavındaki (Val) hatan hiç düşmediyse, demek ki öğrenecek bir şey kalmamıştır. Zaman kaybetme, eğitimi derhal durdur!"
Yani biz 1500 limit koyduk ama o muhtemelen 400.-500. ağaçlarda "Ben optimum noktayı buldum" deyip kendi kendini durduracak.

4. Neden LightGBM? Avantajı ve Dezavantajı Ne?
Terapi Yazılım mülakatının en altın sorusudur. Neden Random Forest, XGBoost veya Derin Öğrenme (LSTM) değil?

Avantajı (Yaprak Odaklı Büyüme - Leaf Wise): XGBoost gibi algoritmalar ağacı kurarken her katmanı simetrik ve düzenli doldurmak zorundadır (Level-wise). Bu yavaşlatır. LightGBM ise asimetrik büyür. Sadece en çok hatayı düşüreceği dalı (yaprağı) bulur ve sadece o tarafa doğru derinleşir. Bu yüzden XGBoost'tan 10 kat daha hızlıdır ve daha az RAM tüketir.

Dezavantajı: Asimetrik büyüdüğü için, eğer elinde küçük bir veri seti (örneğin 5.000 satır) varsa o küçük yapraklara çok hızlı saplanıp aşırı ezber (overfitting) yapar. LightGBM sadece 10.000+ satırlık büyük verilerde kullanılır. (Bizim elimizde 640.000 satır olduğu için en güvenli limandayız).

5. np.expm1(y_val) Muhabbeti Nedir?
Hatırlarsan veriyi temizlerken train_df['sales'] = np.log1p(...) yazıp, satış değerlerinin logaritmasını almıştık. Neden? Çünkü 100 bin satan günler de vardı, 10 satan günler de. Logaritma bu dağ gibi farkları (aykırı değerleri) ezdi, dümdüz pürüzsüz bir tepe yaptı. Modelimiz o pürüzsüz tepeyi (logaritmik sayıları) öğrenerek eğitildi.

Ama model sana tahminde bulunurken logaritmik bir sayı verir. Örneğin 3.2 der.
Sen ERP ekranına bakan şirket müdürüne "Yarın 3.2 adet satacağız" diyemezsin. O 3.2 sayısını matematikte logaritmanın tersi olan Üstel Fonksiyon (Exponential - exp) ile geri çevirmen gerekir.
expm1 (Exponential minus 1), en başta aldığımız log1p (Logarithm plus 1) işleminin tam ve kusursuz geri dönüşümüdür. Modelin uydurduğu rakamı gerçek hayattaki adet sayısına çevirdik ki hatamızı doğru ölçelim.

6. Neden Hata Metriği Olarak MAE ve RMSE Seçildi?
MAE (Ortalama Mutlak Hata): En dürüst, en temiz metriktir. Çıktı olarak 15 verdiyse anlamı şudur: "Senin modelin yarının satışını tahmin ederken ortalama 15 adet yanılıyor." (Yani 100 tahmin ediyorsa gerçekte 85 veya 115 çıkıyor). Şirket yöneticilerinin ve ERP planlamacılarının en sevdiği, en kolay anladığı metriktir.

RMSE (Hata Kareleri Ortalamasının Karakökü): Neden bunu da ekledik? Çünkü RMSE, büyük hataları acımasızca cezalandırır.

Senaryo: Model 9 gün boyunca 1'er adet hata yapsın, 1 gün ise 100 adet hata yapsın. MAE buna bakar ve "Ortalama 10 adet hata yapıyoruz canım, gayet iyi" der. Seni kandırır.

Ama RMSE, o 100 adetlik hatanın karesini alacağı için skoru birden 30'lara fırlatır! "Büyük bir kriz var, bir günü felaket ıskalamışsın!" diye bağırır. Tedarik zincirinde 1 günlük devasa bir sapma bile fabrikada üretimi durdurabileceği için RMSE'yi her zaman kontrol paneline yazarız.

1. MLflow'un Uyanışı (İlk 3 Satır)
INFO mlflow.store.db.utils: Creating initial MLflow database tables...
Experiment with name 'Terapi_Yazilim_ERP_Tahmini' does not exist. Creating a new experiment.

Ne Oldu? MLflow kütüphanesi bilgisayarında ilk defa çalıştığı için kendine gizli bir veritabanı (SQLite) kurdu ve senin verdiğin havalı isimle ("Terapi_Yazilim_ERP_Tahmini") bir proje klasörü açtı. Artık bundan sonra yapacağın tüm eğitimler bu veritabanına kaydedilecek.

2. Modelin Eğitimi ve O Sihirli 1496 Rakamı
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is: [1496]...

Ne Oldu? (Mülakat Şovu): Hatırlarsan modele "Sana 1500 ağaç kurma hakkı veriyorum ama eğer 50 tur boyunca hatan hiç düşmezse eğitimi kes (Early Stopping)" demiştik.

Çıktı bize diyor ki: "Model o kadar hevesli ve verin o kadar kaliteliydi ki, 1500 ağacın 1496'sına gelene kadar her turda öğrenmeye ve hatasını düşürmeye devam etti!" Modelimiz ezberlemeden, kapasitesinin son damlasına kadar öğrenmiş.

3. Asıl Zafer: 🔥 BAŞARI METRİKLERİ 🔥
İşte şirketteki ERP yöneticisinin (veya mülakat komitesinin) bakacağı TEK yer burasıdır.

Doğrulama MAE: 5.35 Adet
Doğrulama RMSE: 6.93 Adet

Bu Ne Demek? İyi mi Kötü mü?

MUHTEŞEM BİR SKOR! Hatırlarsan verimizde ortalama günlük satışlar 50-60 adet civarındaydı. Modelimiz 2017 yılına (geleceğe) bakarak tahmin yaptığında, ortalama sadece 5.35 adet yanılmış. (Yani 50 satacak dediyse, gerçekte 45 veya 55 satmış). ERP ve perakende dünyasında nokta atışına bu kadar yaklaşmak (yaklaşık %10 hata payı) devasa bir başarıdır.

Neden RMSE (6.93) MAE'ye çok yakın? Önceki konuşmamızda "RMSE büyük hataları affetmez, karesini alıp cezalandırır" demiştik. Eğer modelimiz bir gün 50 satacak ürüne gidip 300 tahmin etseydi (büyük bir çuvallama yaşasaydı), o RMSE skoru 6 değil, 30-40 çıkardı! RMSE'nin 6 çıkması, modelimizin hiçbir gün saçmalamadığı, gayet stabil ve güvenilir tahminler yaptığı anlamına gelir.

4. Sarı Uyarı (Warning) ve Kapanış
WARNING mlflow.models.model: artifact_path is deprecated...
✅ Model lokal olarak kaydedildi...

Ne Oldu? O sarı uyarı bir hata değildir. MLflow kütüphanesi sana "Kullandığın bir komutun adı seneye değişecek, haberin olsun" diyor. Hiçbir önemi yok. Önemli olan son satır: Modelin başarıyla eğitildi ve .pkl dosyası olarak kaydedildi.